### Lendo arquivos

In [1]:
import pandas as pd
import os

# Definindo o caminho do arquivo Excel
arquivo_excel = r'dados_bigquery.xlsx'

# Verificando se o arquivo Excel existe
if os.path.exists(arquivo_excel):
    # Lendo a aba "Dados" da planilha Excel
    df_concorrentes = pd.read_excel(arquivo_excel, sheet_name='Dados')
    
    # Convertendo a coluna 'data' para datetime com o formato brasileiro
    df_concorrentes['data'] = pd.to_datetime(df_concorrentes['data'], format='%d/%m/%Y %H:%M:%S', dayfirst=True)
    
    # Adicionando a coluna 'dia_semana'
    df_concorrentes['dia_semana'] = df_concorrentes['data'].dt.day_name()
    
    # Traduzindo os dias da semana para português
    dias_portugues = {
        'Monday': 'Segunda-feira',
        'Tuesday': 'Terça-feira',
        'Wednesday': 'Quarta-feira',
        'Thursday': 'Quinta-feira',
        'Friday': 'Sexta-feira',
        'Saturday': 'Sábado',
        'Sunday': 'Domingo'
    }
    df_concorrentes['dia_semana'] = df_concorrentes['dia_semana'].map(dias_portugues)
    
    print("df_concorrentes carregado com sucesso!")
else:
    print(f"Arquivo Excel não encontrado: {arquivo_excel}")

# Definindo o caminho do arquivo CSV
arquivo_csv = r'linhas_compartilhadas_por_direcao_completo.csv'

# Verificando se o arquivo CSV existe
if os.path.exists(arquivo_csv):
    # Lendo o arquivo CSV
    df_tabela = pd.read_csv(arquivo_csv)
    print("df_tabela carregado com sucesso!")
else:
    print(f"Arquivo CSV não encontrado: {arquivo_csv}")

# Exibindo as primeiras linhas de cada DataFrame
print("\nPrimeiras linhas de df_tabela:")
print(df_tabela.head())
print("\nPrimeiras linhas de df_concorrentes:")
print(df_concorrentes.head())
print("\nDias da semana únicos em df_concorrentes:")
print(df_concorrentes['dia_semana'].unique()) 


df_concorrentes carregado com sucesso!
df_tabela carregado com sucesso!

Primeiras linhas de df_tabela:
  linha_base direcao_base  total_pontos_linha_base linha_compartilhada  \
0        805          Ida                       31              SPB550   
1        805          Ida                       31               SV692   
2        805          Ida                       31               SV692   
3        805          Ida                       31                 343   
4        805          Ida                       31                 343   

  direcao_compartilhada  num_pontos_compartilhados  percentual_cobertura  \
0                   Ida                          1                  3.23   
1                 Volta                          1                  3.23   
2                   Ida                          1                  3.23   
3                 Volta                          2                  6.45   
4                   Ida                          2                  6.4

In [2]:
df_tabela.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3076 entries, 0 to 3075
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   linha_base                 3076 non-null   object 
 1   direcao_base               3076 non-null   object 
 2   total_pontos_linha_base    3076 non-null   int64  
 3   linha_compartilhada        3076 non-null   object 
 4   direcao_compartilhada      3076 non-null   object 
 5   num_pontos_compartilhados  3076 non-null   int64  
 6   percentual_cobertura       3076 non-null   float64
 7   pontos_compartilhados      3076 non-null   object 
dtypes: float64(1), int64(2), object(5)
memory usage: 192.4+ KB


In [3]:
df_concorrentes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17156 entries, 0 to 17155
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   data                   17156 non-null  datetime64[ns]
 1   servico_realizado      17156 non-null  object        
 2   sentido                17156 non-null  object        
 3   quantidade_viagens     16557 non-null  float64       
 4   quantidade_transacoes  17156 non-null  int64         
 5   quantidade_veiculos    16557 non-null  float64       
 6   dia_semana             17156 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 938.3+ KB


### Relatório v5, novo formato

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import Paragraph, Table, TableStyle
from reportlab.lib.units import inch
import os
from pathlib import Path
from PIL import Image
import datetime
from PyPDF2 import PdfMerger
import math


# Definição dos intervalos atualizados (9 intervalos)
import pandas as pd
from datetime import datetime, timedelta
import calendar
from collections import defaultdict

from datetime import datetime, timedelta
import pandas as pd

from datetime import datetime, timedelta
import pandas as pd

def gerar_nove_intervalos_dinamicos():
    # Obter a data atual
    data_atual = datetime.now()
    
    # Encontrar o domingo da semana atual
    ajuste_domingo = data_atual.weekday() + 1  # +1 para converter de 0=segunda para 0=domingo
    if ajuste_domingo == 7:  # Se hoje for domingo
        ajuste_domingo = 0
    domingo_atual = data_atual - timedelta(days=ajuste_domingo)
    domingo_atual = domingo_atual.replace(hour=0, minute=0, second=0, microsecond=0)
    
    # Domingo da semana anterior (este deve ser o último intervalo)
    domingo_anterior = domingo_atual - timedelta(days=7)
    
    # Gerar exatamente 9 intervalos, terminando na semana anterior
    intervalos_raw = []
    for i in range(9):
        inicio = domingo_anterior - timedelta(days=7*(8-i))  # Começa 8 semanas antes da semana anterior
        fim = inicio + timedelta(days=6)
        intervalos_raw.append((inicio, fim))
    
    # Nomes dos meses em português
    nomes_meses = {
        1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril", 5: "Maio", 6: "Junho",
        7: "Julho", 8: "Agosto", 9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
    }
    
    # Agrupar intervalos por mês usando a quinta-feira como referência
    intervalos_por_mes = defaultdict(list)
    
    for inicio, fim in intervalos_raw:
        quinta_feira = inicio + timedelta(days=4)
        mes_da_quinta = quinta_feira.month
        intervalos_por_mes[mes_da_quinta].append((inicio, fim))
    
    # Numerar as semanas dentro de cada mês e criar a lista final
    intervalos_finais = []
    for mes in sorted(intervalos_por_mes.keys()):
        # Determinar quantas semanas deste mês existem no total
        todas_semanas_mes = []
        
        # Obter primeiro dia do mês
        if mes == 1:
            primeiro_dia_mes = datetime(data_atual.year, 1, 1)
        else:
            mes_anterior = mes - 1
            ano = data_atual.year
            ultimo_dia_mes_anterior = datetime(ano, mes_anterior, 1) + timedelta(days=32)
            ultimo_dia_mes_anterior = ultimo_dia_mes_anterior.replace(day=1) - timedelta(days=1)
            primeiro_dia_mes = ultimo_dia_mes_anterior + timedelta(days=1)
        
        # Encontrar domingo que inicia ou antecede o primeiro dia do mês
        ajuste = primeiro_dia_mes.weekday() + 1
        if ajuste == 7:
            ajuste = 0
        primeiro_domingo = primeiro_dia_mes - timedelta(days=ajuste)
        
        # Gerar todas as semanas do mês
        data_temp = primeiro_domingo
        while True:
            quinta = data_temp + timedelta(days=4)
            if quinta.month != mes:
                data_temp += timedelta(days=7)
                continue
            if data_temp > domingo_anterior:
                break
            todas_semanas_mes.append(data_temp)
            data_temp += timedelta(days=7)
        
        # Numerar as semanas do mês que estão em nosso intervalo
        semanas_no_intervalo = sorted(intervalos_por_mes[mes], key=lambda x: x[0])
        for inicio, fim in semanas_no_intervalo:
            # Encontrar o número desta semana no mês
            idx = 1
            for semana_inicio in todas_semanas_mes:
                if semana_inicio == inicio:
                    break
                idx += 1
            
            nome_mes = nomes_meses[mes]
            nome_intervalo = f"{nome_mes} - {idx}ª sem"
            intervalos_finais.append((nome_intervalo, pd.to_datetime(inicio), pd.to_datetime(fim)))
    
    # Print para verificar o resultado
    #print("Intervalos gerados:")
    #for nome, inicio, fim in intervalos_finais:
        #print(f"{nome}: {inicio.strftime('%d/%m/%Y')} - {fim.strftime('%d/%m/%Y')}")
    
    return intervalos_finais

# Gerar os intervalos dinamicamente
intervalos = gerar_nove_intervalos_dinamicos()

# Gerar os intervalos dinamicamente
intervalos = gerar_nove_intervalos_dinamicos()



# Gerar os intervalos dinamicamente
intervalos = gerar_nove_intervalos_dinamicos()

def atribuir_intervalo(data):
    """
    Retorna o rótulo do intervalo no qual a data se encaixa.
    Se a data não estiver em nenhum dos intervalos, retorna "Fora de intervalo".
    """
    for rotulo, inicio, fim in intervalos:
        if inicio <= data <= fim:
            return rotulo
    return "Fora de intervalo"

def mapear_sentido(direcao):
    """
    Mapeia as direções entre os dois DataFrames.
    Converte 'Ida' -> 'I' e 'Volta' -> 'V'
    """
    mapa = {
        'Ida': 'I',
        'Volta': 'V',
        'I': 'Ida',
        'V': 'Volta'
    }
    return mapa.get(direcao, direcao)

def filtrar_dados_concorrentes(df_concorrentes, servico, sentido):
    """
    Filtra o DataFrame pelos critérios especificados e remove dados inválidos.
    """
    # Garantir que servico seja string para comparação consistente
    servico_str = str(servico).strip()
    
    df_filtrado = df_concorrentes[
        (df_concorrentes['servico_realizado'].astype(str).str.strip() == servico_str) &
        (df_concorrentes['sentido'] == sentido)
    ]
    
    # Remove registros com intervalo "Fora de intervalo"
    df_filtrado = df_filtrado[df_filtrado['intervalo'] != "Fora de intervalo"]
    df_filtrado = df_filtrado.dropna(subset=['intervalo'])
    
    return df_filtrado

def calcular_resumo(df_filtrado):
    """
    Calcula o resumo estatístico agrupado por intervalo, incluindo variação percentual.
    """
    # Ordenar os intervalos conforme a sequência definida em 'intervalos'
    ordem_intervalos = {rotulo: i for i, (rotulo, _, _) in enumerate(intervalos)}
    
    # Agrupar por intervalo
    resumo = df_filtrado.groupby('intervalo').agg({
        'quantidade_transacoes': lambda x: x.dropna().mean(),
        'quantidade_viagens': lambda x: x.dropna().mean(),
        'quantidade_veiculos': lambda x: x.dropna().mean()
    }).reset_index()
    
    # Ordenar pelos intervalos definidos
    resumo['ordem'] = resumo['intervalo'].map(ordem_intervalos)
    resumo = resumo.sort_values('ordem')
    
    # Calcular variações percentuais entre semanas consecutivas
    resumo['passageiros'] = resumo['quantidade_transacoes']
    resumo['passageiros_var'] = resumo['quantidade_transacoes'].pct_change(fill_method=None) * 100
    
    resumo['partidas'] = resumo['quantidade_viagens']
    resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change(fill_method=None) * 100
    
    resumo['frota'] = resumo['quantidade_veiculos']
    resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change(fill_method=None) * 100
    
    # Remover colunas de ordem e as originais
    resumo = resumo.drop(columns=['ordem', 'quantidade_transacoes', 'quantidade_viagens', 'quantidade_veiculos'])
    
    return resumo

def preparar_df_concorrentes(df_concorrentes):
    """
    Prepara o DataFrame de concorrentes adicionando a coluna de intervalo.
    """
    # Cria uma cópia para não modificar o original
    df = df_concorrentes.copy()
    
    # Assegura que a coluna 'data' esteja no formato datetime
    df['data'] = pd.to_datetime(df['data'], errors='coerce')
    
    # Cria a coluna 'intervalo' aplicando a função
    df['intervalo'] = df['data'].apply(atribuir_intervalo)
    
    return df

def gerar_tabela_compacta(canvas, titulo, dados_resumo, posicao):
    """
    Gera uma tabela compacta diretamente em um canvas existente.
    Adiciona percentuais de variação entre parênteses.
    
    Args:
        canvas: Canvas do ReportLab para desenhar
        titulo: Título da tabela
        dados_resumo: DataFrame com os dados resumidos
        posicao: Tupla (x, y) da posição na página
    """
    # Verifica se o DataFrame está vazio
    if dados_resumo.empty:
        return
        
    # Limita o número de linhas para garantir que caiba na página
    # Máximo de 10 linhas por tabela para evitar que saia da página
    if len(dados_resumo) > 10:
        dados_resumo = dados_resumo.head(10)
    
    # Prepara os dados para a tabela (sem casas decimais e abreviados)
    tabela_dados = [['Interv.', 'Pass.', 'Part.', 'Frota']]
    
    for _, row in dados_resumo.iterrows():
        # Abrevia os nomes dos intervalos para economizar espaço
        intervalo = row['intervalo']
        intervalo = intervalo.replace('semana', 'sem')
        intervalo = intervalo.replace('Janeiro', 'Jan')
        intervalo = intervalo.replace('Fevereiro', 'Fev')
        intervalo = intervalo.replace('Março', 'Mar')
        intervalo = intervalo.replace('Abril', 'Abr')
        intervalo = intervalo.replace('Maio', 'Mai')
        intervalo = intervalo.replace('Junho', 'Jun')
        intervalo = intervalo.replace('Julho', 'Jul')
        intervalo = intervalo.replace('Agosto', 'Ago')
        intervalo = intervalo.replace('Setembro', 'Set')
        intervalo = intervalo.replace('Outubro', 'Out')
        intervalo = intervalo.replace('Novembro', 'Nov')
        intervalo = intervalo.replace('Dezembro', 'Dez')
        
        # Limita o tamanho do texto do intervalo para 12 caracteres
        if len(intervalo) > 12:
            intervalo = intervalo[:9] + '...'
        
        # Prepara a formatação dos valores com variações percentuais (sem casas decimais)
        passageiros_str = f"{int(row['passageiros']):,}".replace(',', '.') if not pd.isna(row['passageiros']) else "-"
        if not pd.isna(row['passageiros_var']) and math.isfinite(row['passageiros_var']):
            passageiros_str += f" ({int(row['passageiros_var'])}%)"

        # partidas
        partidas_str = f"{int(row['partidas'])}" if not pd.isna(row['partidas']) else "-"
        if not pd.isna(row['partidas_var']) and math.isfinite(row['partidas_var']):
            partidas_str += f" ({int(row['partidas_var'])}%)"

        # frota
        frota_str = f"{int(row['frota'])}" if not pd.isna(row['frota']) else "-"
        if not pd.isna(row['frota_var']) and math.isfinite(row['frota_var']):
            frota_str += f" ({int(row['frota_var'])}%)"
        
        tabela_dados.append([
            intervalo,
            passageiros_str,
            partidas_str,
            frota_str
        ])
    
    # Cria uma tabela com melhor espaçamento entre colunas e coluna de intervalo reduzida
    table = Table(tabela_dados, colWidths=[0.9*inch, 1.0*inch, 0.7*inch, 0.7*inch], spaceBefore=5, spaceAfter=5)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.black),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('ALIGN', (0, 1), (0, -1), 'LEFT'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 8),           # Fonte um pouco maior para legibilidade
        ('FONTSIZE', (0, 1), (-1, -1), 7),          # Fonte um pouco maior para legibilidade
        ('BOTTOMPADDING', (0, 0), (-1, -1), 3),     # Padding um pouco maior
        ('TOPPADDING', (0, 0), (-1, -1), 3),        # Padding um pouco maior
        ('GRID', (0, 0), (-1, -1), 1, colors.black), # Linha da grade mais grossa
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BACKGROUND', (0, 1), (-1, -1), colors.white),
    ]))
    
    # Posição da tabela
    table_x, table_y = posicao
    
    # Garante que a tabela caiba na página (ajusta posição Y se necessário)
    # Obtém as dimensões da tabela
    table_width, table_height = table.wrapOn(canvas, 300, 500)
    
    # Se a tabela for ficar fora da página, ajuste a posição Y
    if table_y - table_height < 30:  # Garante pelo menos 30 pontos de margem inferior
        table_y = 30 + table_height
    
    # Adiciona título da tabela acima dela (com mais espaço)
    canvas.setFont("Helvetica-Bold", 9)  # Fonte um pouco maior para legibilidade
    canvas.drawString(table_x, table_y + 15, titulo)  # 15 pontos acima da tabela
    
    # Desenha a tabela
    table.drawOn(canvas, table_x, table_y - table_height)

def desenhar_lista_linhas(canvas, linhas_para_desenhar, height):
    """
    Desenha uma lista com informações das linhas compartilhadas ao lado da tabela base.
    
    Args:
        canvas: Canvas do ReportLab para desenhar
        linhas_para_desenhar: Lista de tuplas (linha_comp, descricao) a serem desenhadas
        height: Altura da página para posicionamento
    """
    x = 50  # Posição X inicial
    y = height - 80  # Mesma altura da tabela base
    
    # Configura a fonte para os itens da lista
    canvas.setFont("Helvetica", 8)
    
    # Desenha os itens da lista
    for i, (linha_comp, descricao) in enumerate(linhas_para_desenhar):
        # Formato: • linha_compartilhada (descricao_compartilhada)
        texto = f"• {linha_comp}"
        if descricao and not pd.isna(descricao):
            texto += f" ({descricao})"
        
        canvas.drawString(x, y - 15 * i, texto)

def gerar_capa_pdf(output_dir='output', logo_path=None, dia_semana=None):
    """
    Função que gera uma capa em PDF para o relatório de concorrência.
    
    Args:
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        dia_semana: Dia da semana para incluir no título
        
    Returns:
        str: Caminho do arquivo PDF gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"capa_concorrencia.pdf")
    
    # Cria o PDF em orientação retrato (padrão)
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    width, height = letter
    
    # Define a margem padrão
    margin = 40
    
    # Adiciona a logo no centro superior se fornecida
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 1.2  # Fator reduzido ainda mais para logo maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona a logo centralizada no topo
            logo_x = (width - logo_width) / 2
            logo_y = height - logo_height - margin
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
        except Exception as e:
            pass
    
    # Adiciona título principal (aumentado e posicionado mais acima)
    c.setFont("Helvetica-Bold", 28)  # Tamanho aumentado de 24 para 28
    title_y = height / 2 + 80  # Posicionado mais acima (era +50)
    
    # Título com o dia da semana, se fornecido
    if dia_semana:
        c.drawCentredString(width/2, title_y, f"Relatório - Concorrência ({dia_semana})")
    else:
        c.drawCentredString(width/2, title_y, "Relatório - Concorrência")
    
    # Linha horizontal removida conforme solicitado
    
    # Data removida conforme solicitado
    
    # Adiciona informações sobre o relatório
    info_style = ParagraphStyle(
        'Info',
        fontName='Helvetica-Oblique',
        fontSize=11,
        leading=14,
        alignment=1,  # Centralizado
    )
    
    info_text = "Análise comparativa de linhas com pontos compartilhados"
    p = Paragraph(info_text, info_style)
    p.wrapOn(c, width - 2*margin, height)
    p.drawOn(c, margin, title_y - 50)  # Ajustado para ficar mais próximo do título
    
    # Rodapé removido conforme solicitado
    
    # Adiciona número de página
    c.setFont("Helvetica", 8)
    c.drawRightString(width - margin, margin, "Página 1")
    
    # Salva o documento
    c.save()
    
    return pdf_filename

def gerar_pdf_comparacao(df_tabela, df_concorrentes, linha_base=220, direcao_base="Ida", output_dir='output', logo_path=None, pagina_inicial=2, dia_semana=None):
    """
    Função principal que gera um PDF comparando a linha base com suas linhas compartilhadas.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        linha_base: Número da linha base para análise
        direcao_base: Direção da linha base (Ida/Volta)
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        pagina_inicial: Número da primeira página deste relatório (default: 2, considerando a capa como página 1)
        dia_semana: Dia da semana para filtrar os dados (opcional)
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Filtrar df_concorrentes por dia_semana se fornecido
    if dia_semana:
        df_concorrentes = df_concorrentes[df_concorrentes['dia_semana'] == dia_semana].copy()
        if df_concorrentes.empty:
            return None
    
    # Mapear a direção base para o formato do df_concorrentes
    sentido_base = mapear_sentido(direcao_base)
    
    # Preparar o df_concorrentes adicionando a coluna de intervalo
    df_concorrentes_prep = preparar_df_concorrentes(df_concorrentes)
    
    # Obter todas as linhas compartilhadas para esta linha/direção base
    linha_base_str = str(linha_base).strip()
    
    # Usamos .astype(str) para converter todos os valores para string antes de comparar
    linhas_compartilhadas = df_tabela[
        (df_tabela['linha_base'].astype(str).str.strip() == linha_base_str) & 
        (df_tabela['direcao_base'].str.strip() == direcao_base.strip())
    ]
    
    # Verificar se existem linhas compartilhadas
    if linhas_compartilhadas.empty:
        return None
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"comparacao_{linha_base}_{direcao_base.lower()}.pdf")
    
    # Cria o PDF em orientação horizontal
    c = canvas.Canvas(pdf_filename, pagesize=landscape(letter))
    width, height = landscape(letter)
    
    # Define a margem padrão
    margin = 40
    
    # Função auxiliar para adicionar rodapé à página atual
    def adicionar_rodape():
        # Calcular a posição do rodapé estendido até metade da terceira coluna
        rodape_largura = ((width - 3*inch) / 2) + (3*inch / 2) - margin  # Até a metade da terceira coluna
        
        # Criar parágrafo para o rodapé com formatação de negrito para "Nota:"
        rodape_style = ParagraphStyle(
            'Rodape',
            fontName='Helvetica-Oblique',
            fontSize=6,
            leading=8,  # Espaçamento entre linhas
        )
        
        # Usando tags HTML para negrito no texto do rodapé
        rodape_texto = "<b>Nota:</b> Os números de \"Passageiros\", \"Partidas\" e \"Frota\" representam a média diária durante a semana. Os percentuais acima da tabela indicam a cobertura compartilhada da linha em relação à linha base, enquanto os percentuais dentro da tabela mostram a variação em comparação com a semana anterior."
        
        p = Paragraph(rodape_texto, rodape_style)
        p.wrapOn(c, rodape_largura, 30)  # 30pts de altura
        p.drawOn(c, margin, 15)
        
        # Adiciona número de página
        c.setFont("Helvetica", 8)
        c.drawRightString(width - margin, 20, f"Página {page_num}")
    
    # Adiciona a logo no canto superior direito se fornecida
    logo_x = logo_y = logo_width = logo_height = 0
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 3  # Reduzido para logo ainda maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona mais próximo do canto superior direito
            logo_x = width - logo_width - 20  # Reduzido o espaçamento da borda direita
            logo_y = height - logo_height + 15  # Posicionado 15pts acima
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
        except Exception as e:
            pass
    
    # Adiciona um título principal com formato "Comparativo de Linhas (linha_base - direcao_base)"
    c.setFont("Helvetica-Bold", 14)
    c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
    
    # Filtrar dados da linha base
    df_base_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_base, sentido_base)
    
    # Calcular resumo da linha base
    resumo_base = calcular_resumo(df_base_filtrado)
    
    # Posição para a tabela de referência (centralizada no topo)
    base_x = (width - 3*inch) / 2  # Centralizado
    base_y = height - 80  # Conforme solicitado
    
    # Obter o número total de pontos da linha base
    total_pontos = None
    if not linhas_compartilhadas.empty and 'total_pontos_linha_base' in linhas_compartilhadas.columns:
        primeira_linha = linhas_compartilhadas.iloc[0]
        if 'total_pontos_linha_base' in primeira_linha:
            total_pontos = primeira_linha['total_pontos_linha_base']
    
    # Gerar tabela para a linha base na posição de referência
    if total_pontos is not None:
        titulo_base = f"Linha {linha_base} - {direcao_base} ({total_pontos} pontos)"
    else:
        titulo_base = f"Linha {linha_base} - {direcao_base}"
    
    gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
    
    # Reduzindo para 3 tabelas por página (apenas uma linha de tabelas) e movendo para baixo
    positions = [
        # Apenas primeira linha (3 colunas) com posição Y mais baixa
        (margin, height - 320),                   # Esquerda
        ((width - 3*inch) / 2, height - 320),     # Centro
        (width - margin - 3*inch, height - 320),  # Direita
    ]
    
    # Variáveis para controle de página
    page_num = pagina_inicial  # Iniciar com o número de página fornecido
    tabelas_na_pagina = 0
    linhas_na_pagina_atual = []
    
    # Para cada linha compartilhada
    for idx, row in linhas_compartilhadas.iterrows():
        linha_comp = row['linha_compartilhada']
        direcao_comp = row['direcao_compartilhada']
        descricao = row.get('descricao_compartilhada', '')
        
        # Usar percentual_cobertura em vez de percentual_cobertura
        percentual = row['percentual_cobertura']
        
        # Mapear a direção compartilhada para o formato do df_concorrentes
        if pd.isna(direcao_comp) or str(direcao_comp).strip() == "":
            direcao_comp = direcao_base
        
        sentido_comp = mapear_sentido(direcao_comp)
        
        # Filtrar dados da linha compartilhada
        df_comp_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_comp, sentido_comp)
        
        # Verificar se há dados para processar
        if not df_comp_filtrado.empty:
            # Calcular resumo
            resumo_comp = calcular_resumo(df_comp_filtrado)
            
            # Formatação do percentual
            try:
                if not pd.isna(percentual):
                    percentual_float = float(percentual)
                    percentual_formatado = f"{int(percentual_float)}"
                else:
                    percentual_formatado = "N/A"
            except:
                percentual_formatado = str(percentual)
            
            # Título para esta tabela com percentual formatado (sem casas decimais)
            titulo_comp = f"Linha {linha_comp} - {direcao_comp} ({percentual_formatado}%)"
            
            # Adiciona esta linha à lista para a página atual
            linhas_na_pagina_atual.append((linha_comp, descricao))
            
            # Pega a posição atual
            posicao = positions[tabelas_na_pagina]
            
            # Cria a tabela na posição especificada
            gerar_tabela_compacta(c, titulo_comp, resumo_comp, posicao)
            
            # Incrementa contagem de tabelas
            tabelas_na_pagina += 1
            
            # Se completamos 3 tabelas, desenha a lista e cria uma nova página
            if tabelas_na_pagina == 3:
                # Desenha a lista de linhas da página atual
                desenhar_lista_linhas(c, linhas_na_pagina_atual, height)
                
                # Adiciona rodapé e prepara nova página
                adicionar_rodape()
                c.showPage()
                page_num += 1
                
                # Resetar contagens para a próxima página
                tabelas_na_pagina = 0
                linhas_na_pagina_atual = []
                
                # Adiciona cabeçalho na nova página
                c.setFont("Helvetica-Bold", 14)
                c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
                
                # Tenta adicionar a logo novamente
                if logo_path:
                    try:
                        c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
                    except Exception as e:
                        pass
                
                # Adiciona novamente a tabela base na nova página
                gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
    
    # Se ainda temos tabelas na última página, desenha a lista de linhas e o rodapé
    if tabelas_na_pagina > 0:
        desenhar_lista_linhas(c, linhas_na_pagina_atual, height)
        adicionar_rodape()
    
    # Salva o documento
    c.save()
    
    return pdf_filename

def gerar_relatorio_completo_unico(df_tabela, df_concorrentes, output_dir='output', logo_path=None, dia_semana=None):
    """
    Gera um relatório único contendo uma capa e todos os relatórios de comparação.
    As linhas são extraídas automaticamente do df_tabela, mantendo os formatos originais.
    Filtra os dados de concorrentes por dia_semana se fornecido.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        output_dir: Diretório de saída
        logo_path: Caminho para o arquivo da logo
        dia_semana: Dia da semana para filtrar (opcional)
        
    Returns:
        str: Caminho do relatório completo gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Filtrar df_concorrentes por dia_semana se fornecido
    if dia_semana:
        df_concorrentes_filtrado = df_concorrentes[df_concorrentes['dia_semana'] == dia_semana].copy()
        if df_concorrentes_filtrado.empty:
            return None
    else:
        df_concorrentes_filtrado = df_concorrentes.copy()
    
    # Extrair todas as combinações únicas de linha_base e direcao_base
    linhas_direcoes = df_tabela[['linha_base', 'direcao_base']].drop_duplicates().reset_index(drop=True)
    
    # Criar uma coluna para ordenação dos sentidos (Ida = 1, Volta = 2, outros = 3)
    def ordem_sentido(sentido):
        if sentido == 'Ida':
            return 1
        elif sentido == 'Volta':
            return 2
        else:
            return 3
    
    linhas_direcoes['ordem_sentido'] = linhas_direcoes['direcao_base'].apply(ordem_sentido)
    
    # Como linha_base pode conter siglas, vamos manter o formato original e ordenar apenas por sentido
    linhas_direcoes = linhas_direcoes.sort_values(['linha_base', 'ordem_sentido']).reset_index(drop=True)
    
    # Converter para o formato de lista de tuplas
    linhas_base = [(str(row['linha_base']), str(row['direcao_base'])) for _, row in linhas_direcoes.iterrows()]
    
    # Lista para armazenar todos os PDFs temporários gerados
    todos_pdfs = []
    num_pagina_atual = 1
    
    # Primeiro, gerar a capa
    capa_pdf = gerar_capa_pdf(output_dir=output_dir, logo_path=logo_path, dia_semana=dia_semana)
    num_pagina_atual += 1
    todos_pdfs.append(capa_pdf)
    
    # Agora, gerar cada relatório de comparação
    for linha_base, direcao_base in linhas_base:
        # Gerar o PDF de comparação começando na página correta
        pdf_gerado = gerar_pdf_comparacao(
            df_tabela,
            df_concorrentes_filtrado,  # Usar os dados filtrados por dia da semana
            linha_base=linha_base,
            direcao_base=direcao_base,
            output_dir=output_dir,
            logo_path=logo_path,
            pagina_inicial=num_pagina_atual,
            dia_semana=dia_semana  # Passar o dia da semana para a função
        )
        
        if pdf_gerado:
            todos_pdfs.append(pdf_gerado)
            
            # Atualizar o número da próxima página inicial
            # Precisamos determinar quantas páginas foram criadas neste relatório
            try:
                import PyPDF2
                with open(pdf_gerado, 'rb') as f:
                    pdf_reader = PyPDF2.PdfReader(f)
                    num_paginas = len(pdf_reader.pages)
                    num_pagina_atual += num_paginas
            except Exception as e:
                # Supondo que cada relatório tenha ao menos 1 página
                num_pagina_atual += 1
    
    # Combinar todos os PDFs em um único documento
    dia_semana_formatado = dia_semana.replace(" ", "_").lower() if dia_semana else ""
    relatorio_final = os.path.join(output_dir, f"relatorio_completo_concorrencia_{dia_semana_formatado}.pdf")
    
    # Usar PdfMerger para mesclar os PDFs
    merger = PdfMerger()
    
    for pdf in todos_pdfs:
        if os.path.exists(pdf):
            merger.append(pdf)
    
    # Escrever o arquivo final
    merger.write(relatorio_final)
    merger.close()
    
    # Limpar arquivos temporários com força extra
    for pdf in todos_pdfs:
        try:
            if os.path.exists(pdf) and "relatorio_completo" not in pdf:
                os.remove(pdf)
        except Exception:
            try:
                # Segunda tentativa com delay
                import time
                time.sleep(0.5)
                if os.path.exists(pdf):
                    os.remove(pdf)
            except Exception:
                pass
    
    return relatorio_final

# Exemplo de uso
if __name__ == "__main__":
    # df_tabela e df_concorrentes já estão disponíveis no ambiente
    
    # Caminho para a logo
    logo_path = 'Logo_Tijuca.png'
    
    # Obter todos os dias da semana únicos do df_concorrentes
    dias_semana = df_concorrentes['dia_semana'].unique()
    
    # Gerar um relatório para cada dia da semana
    for dia in dias_semana:
        gerar_relatorio_completo_unico(
            df_tabela,
            df_concorrentes,
            logo_path=logo_path,
            dia_semana=dia
        )

In [5]:
def gerar_nove_intervalos_dinamicos():
    # Obter a data atual
    data_atual = datetime.now()
    
    # Encontrar o domingo da semana atual
    ajuste_domingo = data_atual.weekday() + 1  # +1 para converter de 0=segunda para 0=domingo
    if ajuste_domingo == 7:  # Se hoje for domingo
        ajuste_domingo = 0
    domingo_atual = data_atual - timedelta(days=ajuste_domingo)
    domingo_atual = domingo_atual.replace(hour=0, minute=0, second=0, microsecond=0)
    
    # Domingo da semana anterior (este deve ser o último intervalo)
    domingo_anterior = domingo_atual - timedelta(days=7)
    
    # Gerar exatamente 9 intervalos, terminando na semana anterior
    intervalos_raw = []
    for i in range(9):
        inicio = domingo_anterior - timedelta(days=7*(8-i))  # Começa 8 semanas antes da semana anterior
        fim = inicio + timedelta(days=6)
        intervalos_raw.append((inicio, fim))
    
    # Nomes dos meses em português
    nomes_meses = {
        1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril", 5: "Maio", 6: "Junho",
        7: "Julho", 8: "Agosto", 9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
    }
    
    # Agrupar intervalos por mês usando a quinta-feira como referência
    intervalos_por_mes = defaultdict(list)
    
    for inicio, fim in intervalos_raw:
        quinta_feira = inicio + timedelta(days=4)
        mes_da_quinta = quinta_feira.month
        intervalos_por_mes[mes_da_quinta].append((inicio, fim))
    
    # Numerar as semanas dentro de cada mês e criar a lista final
    intervalos_finais = []
    for mes in sorted(intervalos_por_mes.keys()):
        # Determinar quantas semanas deste mês existem no total
        todas_semanas_mes = []
        
        # Obter primeiro dia do mês
        if mes == 1:
            primeiro_dia_mes = datetime(data_atual.year, 1, 1)
        else:
            mes_anterior = mes - 1
            ano = data_atual.year
            ultimo_dia_mes_anterior = datetime(ano, mes_anterior, 1) + timedelta(days=32)
            ultimo_dia_mes_anterior = ultimo_dia_mes_anterior.replace(day=1) - timedelta(days=1)
            primeiro_dia_mes = ultimo_dia_mes_anterior + timedelta(days=1)
        
        # Encontrar domingo que inicia ou antecede o primeiro dia do mês
        ajuste = primeiro_dia_mes.weekday() + 1
        if ajuste == 7:
            ajuste = 0
        primeiro_domingo = primeiro_dia_mes - timedelta(days=ajuste)
        
        # Gerar todas as semanas do mês
        data_temp = primeiro_domingo
        while True:
            quinta = data_temp + timedelta(days=4)
            if quinta.month != mes:
                data_temp += timedelta(days=7)
                continue
            if data_temp > domingo_anterior:
                break
            todas_semanas_mes.append(data_temp)
            data_temp += timedelta(days=7)
        
        # Numerar as semanas do mês que estão em nosso intervalo
        semanas_no_intervalo = sorted(intervalos_por_mes[mes], key=lambda x: x[0])
        for inicio, fim in semanas_no_intervalo:
            # Encontrar o número desta semana no mês
            idx = 1
            for semana_inicio in todas_semanas_mes:
                if semana_inicio == inicio:
                    break
                idx += 1
            
            nome_mes = nomes_meses[mes]
            nome_intervalo = f"{nome_mes} - {idx}ª sem"
            intervalos_finais.append((nome_intervalo, pd.to_datetime(inicio), pd.to_datetime(fim)))
    
    # Print para verificar o resultado
    print("Intervalos gerados:")
    for nome, inicio, fim in intervalos_finais:
        print(f"{nome}: {inicio.strftime('%d/%m/%Y')} - {fim.strftime('%d/%m/%Y')}")
    
    return intervalos_finais

# Gerar os intervalos dinamicamente
intervalos = gerar_nove_intervalos_dinamicos()

Intervalos gerados:
Abril - 1ª sem: 30/03/2025 - 05/04/2025
Abril - 2ª sem: 06/04/2025 - 12/04/2025
Abril - 3ª sem: 13/04/2025 - 19/04/2025
Abril - 4ª sem: 20/04/2025 - 26/04/2025
Maio - 1ª sem: 27/04/2025 - 03/05/2025
Maio - 2ª sem: 04/05/2025 - 10/05/2025
Maio - 3ª sem: 11/05/2025 - 17/05/2025
Maio - 4ª sem: 18/05/2025 - 24/05/2025
Maio - 5ª sem: 25/05/2025 - 31/05/2025
